In [ ]:
from scipy.special import factorial
import pickle
import numpy as np
import pandas as pd
from scipy.special import expit
rng = np.random.default_rng(0)

#### read data

Source data can be downloaded on `https://github.com/prabaey/SimSUM`

In [ ]:
# read source data
data_dir = "./source_data/SimSUM.csv"
df = pd.read_csv(data_dir, delimiter=";", index_col=0)

#### select and process features

In [ ]:
# one hot fever and season
df = pd.get_dummies(df, columns=['fever'], drop_first=False)
df = pd.get_dummies(df, columns=['season'], drop_first=True, prefix='', prefix_sep='')

In [ ]:
%%capture
# binarize text
df = df.replace({'no': 0, 'yes': 1})

# binazire bool
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype('int')

In [ ]:
# rename
df['TEXT'] = df['text']
df['T'] = df['antibiotics']

#### generate treatments and outcomes

In [ ]:
# outcome features
confounders = ['dysp', 'cough', 'pain', 'nasal', 'fever_none', 'fever_low', 'fever_high']
treat_features = confounders + ['policy']
out_features = confounders + ['self_empl']

# ensure format
df[confounders+ ['policy', 'self_empl']] = df[confounders+ ['policy', 'self_empl']].apply(pd.to_numeric, errors='coerce')

In [ ]:
df['M0'] = (
    1.8 * df['self_empl']
    + 1.6 * df['dysp']   
    + 1.2 * df['cough']  
    + 0.5 * df['pain']   
    + 0.4 * df['nasal']  
    + 0.3 * df['fever_low']  
    + 1.5 * df['fever_high'] 
    + 1.1 * df['dysp'] * df['fever_high']  
    + 0.9 * df['self_empl'] * df['cough']  
    - 0.6 * df['nasal'] * df['fever_low']  
    + 0.7 * df['pain'] * df['self_empl']   
    + 0.8 * df['cough'] * df['fever_high'] 
    + 0.7 * df['dysp'] * df['self_empl'])

In [ ]:
logit_e = (
    -2.2
    + 2.0 * df['policy']                    
    + 1.5 * df['dysp']                      
    + 1.1 * df['cough']                     
    + 1.3 * df['fever_high']               
    + 0.6 * df['pain']                     
    + 0.9 * df['policy'] * df['dysp']       
    + 0.7 * df['policy'] * df['fever_high'] 
    - 0.5 * df['nasal'] * df['fever_low'])

df['e'] = np.clip(expit(logit_e), 0.05, 0.95)
df['T'] = rng.binomial(1, df['e'])

In [ ]:
df['cate'] = (
    -1
    + 1.1 * df['pain']
    + 1.0 * df['nasal']
    + 0.7 * df['fever_low']
    - 1.0 * df['dysp']
    - 0.7 * df['cough']
    - 1.1 * df['fever_high'])

df['M1'] = df['M0'] + df['cate']

In [ ]:
sigma_y = 0.8
df['Y0'] = df['M0'] + rng.normal(0, sigma_y, len(df))
df['Y1'] = df['M1'] + rng.normal(0, sigma_y, len(df))
df['Y'] = np.where(df['T']==1, df['Y1'], df['Y0'])

#### store

In [ ]:
# select variables
df = df[['dysp', 'cough', 'pain', 'nasal', 'fever_none', 'fever_low', 'fever_high', 
   'self_empl', 'asthma', 'smoking', 'COPD', 'winter','hay_fever',
   'M0', 'M1', 'Y0', 'Y1', 'cate', 'T', 'Y', 'TEXT']]

In [ ]:
# store
df.to_csv('./datasets/synsum.csv')